# Few-Shot Prompting with the Anthropic SDK

**FEW-SHOT PROMPTING:**
Instead of describing what you want in abstract terms, you SHOW the agent examples of the input/output pattern. Models are excellent pattern matchers. Give them 2-3 examples, and they'll follow the pattern precisely.

**THIS NOTEBOOK DEMONSTRATES:**
1. Defining example input/output pairs
2. Structuring them in a prompt
3. Sending new input and getting pattern-matched output
4. Comparing zero-shot vs few-shot results

**KEY TAKEAWAY:**
Few-shot examples eliminate ambiguity. Instead of saying "format it nicely," you show what "nicely" means with concrete examples.

## Setup

Install the Anthropic SDK and initialize the client.

In [ ]:
!pip install anthropic

In [ ]:
import os
import json
import anthropic

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

## Example 1: Few-Shot for Structured Data Extraction

**Task:** Extract property details from unstructured listing descriptions into a consistent JSON format.

Without few-shot examples, the agent might return different JSON structures each time. With examples, it follows the exact pattern.

In [ ]:
EXTRACTION_EXAMPLES = [
    {
        "input": (
            "Beautiful 3 bed 2 bath home in Kailua with stunning ocean views. "
            "Recently renovated kitchen and bathrooms. 1,450 sq ft of living space. "
            "Listed at $1.2M. Large backyard perfect for entertaining."
        ),
        "output": {
            "address_area": "Kailua",
            "bedrooms": 3,
            "bathrooms": 2,
            "sqft": 1450,
            "price": 1200000,
            "key_features": ["ocean views", "renovated kitchen", "renovated bathrooms", "large backyard"],
            "property_type": "single_family",
        },
    },
    {
        "input": (
            "Spacious 2BR/2BA condo in Waikiki, unit 1805. Floor-to-ceiling windows "
            "with Diamond Head views. Building has pool, gym, and 24hr security. "
            "875 square feet. Asking $680,000. HOA includes water and cable."
        ),
        "output": {
            "address_area": "Waikiki",
            "bedrooms": 2,
            "bathrooms": 2,
            "sqft": 875,
            "price": 680000,
            "key_features": ["Diamond Head views", "pool", "gym", "24hr security", "HOA includes water and cable"],
            "property_type": "condo",
        },
    },
    {
        "input": (
            "Charming cottage on the North Shore, Haleiwa. 1 bedroom, 1 bath, "
            "just 550 sqft but steps from the beach. Great vacation rental income "
            "potential. Price: $495K. Comes fully furnished."
        ),
        "output": {
            "address_area": "Haleiwa",
            "bedrooms": 1,
            "bathrooms": 1,
            "sqft": 550,
            "price": 495000,
            "key_features": ["steps from beach", "vacation rental potential", "fully furnished"],
            "property_type": "single_family",
        },
    },
]

### Building the Few-Shot Prompt

The structure is:
1. Task description
2. Example 1: input -> output
3. Example 2: input -> output
4. Example 3: input -> output
5. New input (for the model to process)

The model sees the pattern and follows it exactly.

In [ ]:
def build_few_shot_prompt(examples: list[dict], new_input: str) -> str:
    """
    Build a prompt with few-shot examples.

    The structure is:
    1. Task description
    2. Example 1: input -> output
    3. Example 2: input -> output
    4. Example 3: input -> output
    5. New input (for the model to process)

    The model sees the pattern and follows it exactly.
    """
    prompt = "Extract property details from listing descriptions into structured JSON.\n\n"

    # Add each example
    for i, example in enumerate(examples, 1):
        prompt += f"Example {i}:\n"
        prompt += f"Input: {example['input']}\n"
        prompt += f"Output: {json.dumps(example['output'], indent=2)}\n\n"

    # Add the new input for the model to process
    prompt += f"Now extract from this listing:\n"
    prompt += f"Input: {new_input}\n"
    prompt += f"Output:"

    return prompt


def extract_property_data(listing_text: str) -> dict:
    """
    Use few-shot prompting to extract structured data from a listing description.
    """
    prompt = build_few_shot_prompt(EXTRACTION_EXAMPLES, listing_text)

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}],
    )

    # Parse the JSON response
    response_text = response.content[0].text.strip()
    return json.loads(response_text)

## Example 2: Few-Shot for Consistent Tone/Style

**Task:** Write property listing descriptions in a consistent voice. Few-shot examples establish the tone, length, and style.

Notice the examples are structured as alternating user/assistant messages. This is another way to do few-shot: put the examples in the conversation history so the model sees "when the user asks X, I respond with Y."

In [ ]:
DESCRIPTION_SYSTEM_PROMPT = """You write short property listing descriptions for Hawaii real estate.
Follow the exact style shown in the examples. Keep descriptions to 2-3 sentences."""

DESCRIPTION_EXAMPLES = [
    {
        "role": "user",
        "content": "Write a listing description: 3BR/2BA, Kailua, 1400 sqft, ocean view, renovated",
    },
    {
        "role": "assistant",
        "content": (
            "Stunning 3-bedroom, 2-bath Kailua gem with sweeping ocean views that will "
            "take your breath away. Freshly renovated throughout with 1,400 sq ft of "
            "thoughtfully designed living space. This is coastal Hawaii living at its finest."
        ),
    },
    {
        "role": "user",
        "content": "Write a listing description: 2BR/1BA, Manoa, 900 sqft, garden view, quiet street",
    },
    {
        "role": "assistant",
        "content": (
            "Peaceful 2-bedroom, 1-bath Manoa retreat nestled on a quiet, tree-lined street. "
            "Lush garden views from every window complement the charming 900 sq ft interior. "
            "A rare find in one of Honolulu's most sought-after valley neighborhoods."
        ),
    },
]


def generate_listing_description(property_details: str) -> str:
    """
    Generate a listing description using few-shot conversation examples.

    Notice the examples are structured as alternating user/assistant messages.
    This is another way to do few-shot: put the examples in the conversation
    history so the model sees "when the user asks X, I respond with Y."
    """
    # Start with the few-shot examples as conversation history
    messages = DESCRIPTION_EXAMPLES.copy()

    # Add the new request
    messages.append({
        "role": "user",
        "content": f"Write a listing description: {property_details}",
    })

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=200,
        system=DESCRIPTION_SYSTEM_PROMPT,
        messages=messages,
    )

    return response.content[0].text

## Example 3: Few-Shot for Classification

**Task:** Classify buyer leads by urgency level. Few-shot examples define what each category looks like.

In [ ]:
CLASSIFICATION_PROMPT = """Classify buyer leads into urgency categories.

Example 1:
Lead: "We're moving to Hawaii in 2 months for my husband's job. Need to find a 3BR near Pearl Harbor. Pre-approved for $650K."
Classification: HOT
Reason: Firm timeline, specific needs, financially ready

Example 2:
Lead: "My wife and I are thinking about retiring to Hawaii in a few years. Just starting to look at what's available."
Classification: WARM
Reason: Genuine interest but no timeline or financial preparation mentioned

Example 3:
Lead: "I saw your listing on Zillow. How's the Hawaii market doing? I might invest someday."
Classification: COLD
Reason: Vague interest, no specific intent, no timeline

Example 4:
Lead: "We just sold our home on the mainland and close escrow in 30 days. Looking to buy in Kailua or Kaneohe, budget is $1.1M. Can we see homes this weekend?"
Classification: HOT
Reason: Active buyer, funded, specific areas, requesting immediate action

Now classify this lead:
Lead: "{lead_text}"
Classification:"""


def classify_lead(lead_text: str) -> dict:
    """
    Classify a buyer lead using few-shot examples.
    Returns the classification and reasoning.
    """
    prompt = CLASSIFICATION_PROMPT.format(lead_text=lead_text)

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=150,
        messages=[{"role": "user", "content": prompt}],
    )

    return response.content[0].text.strip()

## Demo: Structured Data Extraction

The few-shot prompt sends 3 examples of input->JSON extraction, then asks the model to extract from a new listing.

In [ ]:
print("=" * 60)
print("FEW-SHOT PROMPTING EXAMPLES")
print("=" * 60)

print("\n--- 1. STRUCTURED DATA EXTRACTION ---\n")

test_listing = (
    "Move-in ready 4BR/2.5BA in Hawaii Kai with marina views and private dock. "
    "Open floor plan with 2,100 sq ft. Priced at $1.45 million. "
    "Solar panels, EV charger, and tropical landscaping throughout."
)

print(f"Input listing:\n{test_listing}\n")
print("Few-shot prompt sends 3 examples of input->JSON extraction")
print("Then asks the model to extract from the new listing.\n")

# In a live run with API key, uncomment:
# result = extract_property_data(test_listing)
# print(f"Extracted data:\n{json.dumps(result, indent=2)}")

# Show what the expected output would look like:
expected = {
    "address_area": "Hawaii Kai",
    "bedrooms": 4,
    "bathrooms": 2.5,
    "sqft": 2100,
    "price": 1450000,
    "key_features": ["marina views", "private dock", "solar panels", "EV charger", "tropical landscaping"],
    "property_type": "single_family",
}
print(f"Expected output pattern:\n{json.dumps(expected, indent=2)}")

## Demo: Consistent Listing Descriptions

The few-shot examples establish the tone and length. The model will match the style of the examples.

In [ ]:
print("\n--- 2. CONSISTENT LISTING DESCRIPTIONS ---\n")

property_details = "4BR/3BA, Hawaii Kai, 2100 sqft, marina view, solar panels"
print(f"Input: {property_details}\n")
print("Few-shot examples establish the tone and length.")
print("The model will match the style of the examples.\n")

# In a live run:
# description = generate_listing_description(property_details)
# print(f"Generated:\n{description}")

print("Expected style: 2-3 sentences, warm but professional, highlights key features")

## Demo: Lead Classification

The few-shot examples teach the model what HOT, WARM, and COLD look like. It applies the same criteria to new leads consistently.

In [ ]:
print("\n--- 3. LEAD CLASSIFICATION ---\n")

test_leads = [
    "Hi, I'm relocating from California next month. Pre-approved for $900K. Looking for a 3BR in Kailua near the schools. When can we tour?",
    "Aloha! My partner and I love Hawaii. We visited last summer and dream of owning a place there. Maybe in 5 years when we retire.",
    "Saw your ad. What's the cheapest house in Hawaii?",
]

for lead in test_leads:
    print(f'Lead: "{lead}"')
    # In a live run:
    # classification = classify_lead(lead)
    # print(f"Result: {classification}")
    print()

print("The few-shot examples teach the model what HOT, WARM, and COLD look like.")
print("It applies the same criteria to new leads consistently.")

## Key Takeaways

1. Few-shot examples eliminate ambiguity in output format
2. 2-5 examples is usually sufficient (diminishing returns after that)
3. Examples can go in the prompt text OR as conversation history
4. Make examples representative of real data and edge cases
5. Consistent examples = consistent outputs